# 03 — Churn Prediction Model

This notebook builds two simple classification models, compares them using standard metrics, and produces customer-level churn risk scores for the reporting layer.

## 1. Load cleaned data

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

df = pd.read_csv('../data/telco_churn_cleaned.csv')
X = df.drop(columns=['customerID','Churn','Churn_flag','tenure_bucket'])
y = df['Churn_flag']


## 2. Prepare features

In [2]:
categorical_features = X.select_dtypes(include='object').columns.tolist()
numeric_features = X.select_dtypes(exclude='object').columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (5634, 19) Test: (1409, 19)


/tmp/ipykernel_156/3459480584.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include='object').columns.tolist()


## 3. Logistic Regression

In [3]:
logistic_model = Pipeline([('preprocessor', preprocessor), ('model', LogisticRegression(max_iter=1000))])
logistic_model.fit(X_train, y_train)
log_pred = logistic_model.predict(X_test)
log_prob = logistic_model.predict_proba(X_test)[:,1]

## 4. Random Forest

In [4]:
rf_model = Pipeline([('preprocessor', preprocessor), ('model', RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'))])
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:,1]

## 5. Compare model performance

In [5]:
def score_model(name, y_true, pred, prob):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, pred),
        'Precision': precision_score(y_true, pred),
        'Recall': recall_score(y_true, pred),
        'F1': f1_score(y_true, pred),
        'ROC_AUC': roc_auc_score(y_true, prob)
    }

results = pd.DataFrame([score_model('Logistic Regression', y_test, log_pred, log_prob), score_model('Random Forest', y_test, rf_pred, rf_prob)])
results.round(3)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.806,0.657,0.559,0.604,0.842
1,Random Forest,0.784,0.621,0.473,0.537,0.823


## 6. Confusion matrix for Logistic Regression

In [6]:
print(confusion_matrix(y_test, log_pred))

[[926 109]
 [165 209]]


## 7. Create customer-level risk scores

In [7]:
all_X = df.drop(columns=['customerID','Churn','Churn_flag','tenure_bucket'])
df['churn_risk_score'] = logistic_model.predict_proba(all_X)[:,1]
df['predicted_high_risk'] = (df['churn_risk_score'] >= 0.50).astype(int)

risk_list = df[['customerID','tenure','Contract','MonthlyCharges','InternetService','Churn','churn_risk_score','predicted_high_risk']].sort_values('churn_risk_score', ascending=False)
risk_list.head(10)

,customerID,tenure,Contract,MonthlyCharges,InternetService,Churn,churn_risk_score,predicted_high_risk
1976,9497-QCMMS,1,Month-to-month,93.55,Fiber optic,Yes,0.855494,1
4800,9300-AGZNL,1,Month-to-month,94.00,Fiber optic,Yes,0.854411,1
3380,5178-LMXOP,1,Month-to-month,95.10,Fiber optic,Yes,0.854280,1
3749,4424-TKOPW,2,Month-to-month,93.85,Fiber optic,Yes,0.851391,1
6368,2720-WGKHP,2,Month-to-month,94.00,Fiber optic,Yes,0.850569,1
5989,5567-WSELE,3,Month-to-month,94.60,Fiber optic,Yes,0.848054,1
1410,7024-OHCCK,2,Month-to-month,93.85,Fiber optic,Yes,0.847839,1
3159,5150-ITWWB,3,Month-to-month,94.85,Fiber optic,No,0.846503,1
2208,7216-EWTRS,1,Month-to-month,100.80,Fiber optic,Yes,0.844837,1
997,1374-DMZUI,4,Month-to-month,94.30,Fiber optic,Yes,0.841069,1


## 8. Export the model output

In [8]:
df.to_csv('../data/model_predictions.csv', index=False)
results.to_csv('../docs/model_metrics.csv', index=False)
print('Saved model predictions and metrics.')

Saved model predictions and metrics.
